# [실습] LangChain으로 첫 체인 만들기

## 라이브러리 설치  

랭체인 관련 라이브러리를 설치합니다.  


In [5]:
# LangChain + LLM.......$@%^#*
!pip install langchain-huggingface langchain langchain-openai langchain-community langchain-ollama langchain-core langchain-google-genai #langchain-google-generativeai
!pip install dotenv langchain-chroma

## 환경 변수 설정하기


In [6]:
import os
from dotenv import load_dotenv
# OPENAI_API_KEY, GOOGLE_API_KEY
load_dotenv(".env", override=True)

True

모델 Provide의 클래스를 불러온 뒤, LLM을 연결하면 됩니다!

## Ollama를 통해 LLM 불러오기


https://ollama.com/


올라마는 가장 간단하게 오픈 모델을 구동할 수 있는 방식입니다.   
추론 최적화/병렬 처리 등의 기능은 뛰어나지 않지만  
CPU에서도 구동이 가능하며, UI도 지원합니다.

### Ollama 설치하고 서빙하기    
터미널을 이용해 아래의 커맨드를 입력합니다.

- 설치

```curl -fsSL https://ollama.com/install.sh | sh```

- Context Window 늘리기 (기본 Context 4096)

```export OLLAMA_CONTEXT_LENGTH=16384```

- 서빙 (& 으로 백그라운드 실행)

``` ollama serve &```



In [ ]:
#!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd


In [7]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 57 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (403 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122579 files and directories currently i

In [8]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [9]:
!ollama --version
!export OLLAMA_CONTEXT_LENGTH=16384
#!ollama serve &

올라마는 기본적으로 11434 포트에서 실행됩니다.   
Pull 을 통해 허깅페이스 모델의 주소나 Ollama 주소를 입력하면 모델을 불러와 저장합니다.

불러올 수 있는 모델의 목록은 https://ollama.com/search 에서 확인할 수 있습니다.   

기본적으로 양자화를 거친 상태로 불러오며, 16비트 Full Precision을 비롯한 다양한 버전도 확인할 수 있습니다.

터미널에서 아래 커맨드를 입력해 GPT-OSS 20B 모델을 불러옵니다.    
```
ollama pull gpt-oss:20b
```

In [ ]:
#!ollama pull gpt-oss:20b

Error: could not connect to ollama server, run 'ollama serve' to start it


In [10]:
import subprocess
import time

ollama_process = subprocess.Popen(["ollama", "serve"],
                                  stdout=subprocess.PIPE,
                                  stderr=subprocess.PIPE
                                  )


time.sleep(10)
print("Ollama server started")

Ollama server started


In [11]:
!ollama pull gpt-oss:20b

불러온 모델은 파일 시스템에 저장되며,   
Ollama 모듈을 호출할 때 해당 모델이 GPU에 로드되는 구조입니다.

In [12]:
#import sys
#!{sys.executable} -m pip install langchain-ollama
from langchain_ollama import ChatOllama
llm = ChatOllama(model='gpt-oss:20b',
                 temperature=0.1, reasoning=True)

test = '안녕!!!!!'
llm.invoke(test)

AIMessage(content='안녕하세요! 😊 오늘은 어떤 도움을 드릴까요?', additional_kwargs={'reasoning_content': 'User says "안녕!!!!!" which is Korean for "Hello!!!!!". They greet. We should respond in Korean, friendly. Probably "안녕하세요! 어떻게 도와드릴까요?" or something. Let\'s respond warmly.'}, response_metadata={'model': 'gpt-oss:20b', 'created_at': '2026-09-05T11:22:26.606010869Z', 'done': True, 'done_reason': 'stop', 'total_duration': 107040344742, 'load_duration': 65521070734, 'prompt_eval_count': 70, 'prompt_eval_duration': 33211381999, 'eval_count': 68, 'eval_duration': 8293324000, 'logprobs': None, 'model_name': 'gpt-oss:20b', 'model_provider': 'ollama'}, id='lc_run--01a0714c-958b-7e50-9866-f2bdf91693fb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 70, 'output_tokens': 68, 'total_tokens': 138})

In [13]:
test = 'ㅠㅠ'
for s in llm.stream(test):
    print(s.content, end='')

아이고, 무슨 일 있었나요? 😢 혹시 얘기하고 싶으신 게 있으면 언제든지 말씀해 주세요. 제가 도와드릴 수 있는 부분이 있다면 최선을 다해 도와드릴게요. 🌸

## Embedding


RAG의 구조는 검색을 수행하기 위한 임베딩 모델이 필요합니다.   
임베딩 모델은 텍스트를 벡터로 변환하며,    
이후 결과를 벡터 DB에 저장해 검색할 수 있습니다.

OpenAI의 `text-embedding-3-large` 는 빠른 속도로 연산이 가능하나, 비용이 발생하며 온라인 모델입니다.   
이에 따라, 폐쇄망/온프레미스 환경에서는 공개 임베딩 모델을 사용하여 구현해야 합니다.

In [14]:
from langchain_openai import OpenAIEmbeddings
openai_embeddings = OpenAIEmbeddings(model='text-embedding-3-large')

허깅페이스에 게시된 공개 모델을 불러옵니다.   
오픈 임베딩 모델에서 중요한 파라미터는 다음과 같습니다.

- 파라미터 수 : 큰 임베딩 모델의 크기는 LLM에 육박합니다. GPU를 고려하여 선택합니다.
- Max Tokens: 임베딩 모델의 최대 토큰보다 큰 데이터를 입력하면, 앞부분만을 이용해 계산하게 되므로 적절한 검색이 되지 않을 수 있습니다.
- 임베딩 차원: 큰 차원의 벡터를 생성하는 임베딩 모델은 검색 속도가 감소합니다.

현재 한국어 데이터를 임베딩하기 위해 자주 사용하는 모델은 아래와 같습니다.

- BAAI/BGE-M3 (2GB, 8194 토큰 제한)   
BGE-M3 시리즈는 BAAI의 임베딩 모델로, 현재 가장 인기가 많은 모델입니다.

- nlpai-lab/KURE-v1 (2GB, 8194 토큰 제한)    
KURE 임베딩은 고려대학교 NLP 연구실에서 만든 모델로, BGE-M3를 한국어 텍스트로 파인 튜닝한 모델입니다.

- Qwen/Qwen-3-Embedding (0.6B, 4B, 8B, 32768 토큰 제한)    
알리바바 클라우드의 Qwen 모델을 개량하여 만든 모델입니다.    
가장 최신 모델로, BGE-M3와의 성능 비교가 치열합니다.

In [15]:
from sentence_transformers import SentenceTransformer
import torch

# HuggingFace 임베딩 주소 지정하기
# intfloat/multilingual-e5-small , baai/bge-m3, 등의 주소를 입력하여 지정
# GPU에 여유가 있다면 Qwen3 Embedding의 큰 사이즈 (4B, 8B)?

#model_name = 'Qwen/Qwen3-Embedding-0.6B'
model_name = 'Qwen/Qwen3-Embedding-4B'
#실제 주소: https://huggingface.co/BAAI/bge-m3

# CPU 설정으로 모델 불러오기
emb_model = SentenceTransformer(model_name,
            device='cuda', #'cpu',
            model_kwargs={'torch_dtype':torch.bfloat16})

# 로컬 폴더에 모델 저장하기
emb_model.save('./embedding')

# 모델 메모리에서 삭제
del emb_model
import gc
gc.collect()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/30.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

... (6 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

9548

파일 시스템에 저장한 오픈 모델은 HuggingFaceEmbeddings로 불러옵니다.

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings
import torch

# 허깅페이스 포맷의 임베딩 모델 불러오기
open_embeddings = HuggingFaceEmbeddings(model_name= './embedding',
                                   model_kwargs={'device':'cuda', 'model_kwargs':{'torch_dtype':torch.bfloat16}})
                                                    # gpu 사용하기                               bfloat16

# GPU 로드된 것 확인
print('임베딩 모델 로드 완료!')

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

임베딩 모델 로드 완료!


RAG를 하기 전, 비교를 위해 LLM에게 질문해 보겠습니다.

In [17]:
# Test
thinking = False
for s in llm.stream("도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?"):
    if s.additional_kwargs.get('reasoning_content'):
        if not thinking:
            print('<THINK> ', end='')
            thinking=True
        print(s.additional_kwargs['reasoning_content'], end='')
    else:
        if thinking:
            print('</THINK>\n\n', end='')
            thinking=False
        print(s.content,end='')

<THINK> The user asks in Korean: "도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?" Means: "What is a domain-specific language model? What are some examples?" They want explanation of domain-specific language models (DSLMs). Provide definition, characteristics, examples like medical domain models (BioBERT, ClinicalBERT), legal domain models, finance, code, etc. Also mention specialized training, fine-tuning, domain adaptation. Provide examples: PubMedBERT, SciBERT, BioGPT, CodeBERT, GraphCodeBERT, LegalBERT, FinBERT, etc. Also mention specialized models like GPT-4 fine-tuned for legal, medical, etc. Provide explanation of how they differ from general-purpose models. Provide examples of tasks: named entity recognition, question answering, summarization, etc. Provide references. Provide Korean explanation. Provide examples of domain-specific language models. Provide mention of open-source models. Provide mention of domain adaptation via pretraining on domain corpora. Provide mention of specialized toke

## 데이터 준비하기    
네이버 API를 통해, 검색어에 대한 뉴스 기사 링크를 가져오겠습니다.    


In [18]:
import requests
import os

def get_naver_news_links(query, num_links=100):
    """
    query와 num_links를 입력받아 네이버 검색 수행, 네이버 뉴스 URL의 기사만 수집
    """

    #url = f"https://openapi.naver.com/v1/search/news.json?query={query}&display={num_links}&sort=sim"
    url = f"https://naverapihub.apigw.ntruss.com/search/v1/news?query={query}&display={num_links}&start=1&sort=date&format=json"
    # 최대 100개의 결과를 표시

    client_id = os.getenv('X-NCP-APIGW-API-KEY-ID')##('NAVER_CLIENT_ID')
    client_secret = os.getenv('X-NCP-APIGW-API-KEY')##('NAVER_CLIENT_SECRET')

    headers = {
        # 'X-Naver-Client-Id': client_id,
        # 'X-Naver-Client-Secret': client_secret
        'X-NCP-APIGW-API-KEY-ID' : client_id,
        'X-NCP-APIGW-API-KEY' : client_secret
    }

    response = requests.get(url, headers=headers)
    result = response.json()

    # 특정 링크 형식만 필터링
    filtered_links = []
    for item in result['items']:
        link = item['link']
        if "n.news.naver.com/mnews/article/" in link:
            # 네이버 뉴스 스타일만 모으기
            filtered_links.append(link)

    # 결과 출력
    #print(query, ':', len(filtered_links))
    if filtered_links :
       example = filtered_links[0]
    else :
       example = None
    print(query, ':', len(filtered_links), 'Example:', example)
    # for link in filtered_links:
    #     print(link)

    return filtered_links

filtered_links = []
for topic in ['도메인 특화 언어모델', 'OpenAI', 'GPT', '구글', '가전제품', '넷플릭스']:
    filtered_links += get_naver_news_links(topic, 100)
print('Total Articles:', len(filtered_links))
print('Total Articles(Without Duplicate):',len(list(set(filtered_links))))
filtered_links = list(set(filtered_links))

도메인 특화 언어모델 : 30 Example: https://n.news.naver.com/mnews/article/092/0002436279?sid=110
OpenAI : 10 Example: https://n.news.naver.com/mnews/article/009/0005729918?sid=104
GPT : 36 Example: https://n.news.naver.com/mnews/article/015/0005329025?sid=101
구글 : 34 Example: https://n.news.naver.com/mnews/article/015/0005329025?sid=101
가전제품 : 37 Example: https://n.news.naver.com/mnews/article/657/0000055047?sid=102
넷플릭스 : 31 Example: https://n.news.naver.com/mnews/article/011/0004658884?sid=102
Total Articles: 178
Total Articles(Without Duplicate): 172


## LangChain Document Loaders

LangChain의 `document_loaders`는 다양한 형식의 파일을 불러올 수 있습니다.   
[https://python.langchain.com/docs/integrations/document_loaders/ ]    

Web URL로부터 페이지를 로드하는 기본 파서인 `WebBaseLoader`를 사용합니다.   

In [19]:
# Jupyter 분산 처리를 위한 설정
import nest_asyncio

nest_asyncio.apply()

In [20]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

async def get_news_documents(links):
    loader = WebBaseLoader(
        web_paths=links,
        bs_kwargs={'parse_only':bs4.SoupStrainer(class_=("newsct", "newsct-body"))},
                                # newsct, newsct-body만 추출 : 네이버 뉴스 포맷 HTML 요소

        requests_per_second = 10, # 1초에 10개 요청 보내기
        show_progress = True # 진행 상황 출력
    )
    # docs = loader.load() # 기본 코드
    # return docs

    docs = []

    async for doc in loader.alazy_load():
        # 순차적 로드 대신 비동기 처리
        docs.append(doc)
    return docs

docs = await get_news_documents(filtered_links)

/tmp/ipykernel_1395/1843925440.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
Fetching pages: 100%|##########| 172/172 [00:03<00:00, 48.96it/s]


In [21]:
from rich import print as rprint
rprint(docs[12])

Document(
    metadata={'source': 'https://n.news.naver.com/mnews/article/366/0001190534?sid=102'},
    page_content='\n\n\n\n\n\n\n조선비즈\n\n\n구독\n\n조선비즈 언론사 구독되었습니다. 메인 뉴스판에서  주요뉴스를  
볼 수 있습니다.\n보러가기\n닫기\n\n\n조선비즈 언론사 구독 해지되었습니다.\n닫기\n\n\n\n\nPICK\n안내\n\n\n언론사가 
주요기사로선정한 기사입니다.\n언론사별 바로가기\n닫기\n\n\n\n\n5일 여의도 밤하늘 밝힌다…제22회 
서울세계불꽃축제\n\n\n\n\n\n\n\n\n입력\n2026.09.05. 오전 9:47\n\n\n수정\n2026.09.05. 오전 9:54\n\n\n\n기사원문\n 
\n\n\n\n\n\n\n\n\n\n\n추천\n반응\n\n\n\n\n쏠쏠정보\n0\n\n\n\n\n흥미진진\n0\n\n\n\n\n공감백배\n0\n\n\n\n\n분석탁월\n
0\n\n\n\n\n후속강추\n0\n\n\n \n\n\n\n\n댓글\n반응\n\n\n\n\n\n\n\n\n텍스트 음성 변환 서비스 
사용하기\n\n\n\n성별\n남성\n여성\n\n\n말하기 속도\n느림\n보통\n빠름\n\n이동 통신망을 이용하여 음성을 재생하면 
별도의 데이터 통화료가 부과될 수 있습니다.\n본문듣기 시작\n\n닫기\n\n\n \n\n글자 크기 
변경하기\n\n글자크기\n\n\n가1단계\n작게\n\n\n가2단계\n보통\n\n\n가3단계\n크게\n\n\n가4단계\n아주크게\n\n\n가5단계\n
최대크게\n\n\n닫기\n\n\n\n\nSNS 보내기\n\n\n\n인쇄하기\n\n\n\n\n\n\n\n\n\n\n\n\n서울세계불꽃축제 2026을 하루 앞둔 
4일 서울 영등포구 여의도 한강공원 일대에서 열린 \'프리덤 250 기념 불꽃쇼 및 드론쇼\'에서 불꽃이 터지고 있다. 
/연합뉴스         가을밤 서울의 하늘이 다시 한번 화려한 빛으로 물든다. 한화그룹의 ‘서울세계불꽃축제’가 5일 여의도 
한강공원 일대에서 열린다.올해로 22회째를 맞은 축제는 오후 8시 정각 시작한다. ‘당신의 빛을 찾아서’를 주제로 한국 
한화를 비롯해 미국과 영국 등 3개국 대표 불꽃팀이 차례로 밤하늘에 각국의 색깔을 펼친다.첫 주자는 영국팀이다. 
넷플릭스 애니메이션 ‘케이팝 데몬 헌터스’ 삽입곡 ‘소다팝’ 등에 맞춰 특별 제작한 불꽃을 선보이며 축제의 문을 연다. 
이어 미국팀은 ‘카르미나 부라나: 오 포르투나’ 등의 웅장한 선율에 맞춰 어둠을 가르는 빛의 장관을 연출한다.한화의 
공연은 오후 8시 40분부터 시작된다. 고흐의 ‘별이 빛나는 밤’에서 영감을 얻은 불꽃쇼부터 가수 이소라의 ‘바람이 분다’에
맞춘 감성적인 연출까지 음악과 불꽃이 어우러진 무대를 선보인다.올해 불꽃축제의 관전 포인트는 원효대교다. 원효대교를 
활용한 ‘타워불꽃’과 한층 다양해진 ‘브릿지 연출’이 더해진다. 특히 원효대교를 중심으로 한강 상공 양쪽에서 불꽃이 
대칭적으로 펼쳐지는 ‘데칼코마니’ 연출은 축제의 시그니처로 꼽힌다. 멀리서도 넓은 시야로 불꽃을 즐길 수 있도록 
와이드한 연출을 강화했다.100만명 안팎의 인파가 몰리는 대형 행사인 만큼 안전관리에도 힘을 쏟는다.한화는 임직원 
봉사단 1200명을 포함해 역대 최대 규모인 5000여명을 현장에 투입한다. 실시간 인파 밀집도를 측정하는 ‘오렌지세이프티’ 
시스템과 KT의 5G 네트워크 슬라이싱 기술을 활용해 현장 상황을 살핀다. 서울시·소방재난본부·서울경찰청 등 관계기관과 
종합상황실도 운영한다.직접 여의도를 찾지 못하는 시민을 위한 온라인 관람도 가능하다. 한화는 공식 유튜브 채널 
‘한화TV’를 통해 여의도 불꽃쇼를 실시간으로 중계할 예정이다.\n\n\n\n\nCopyright ⓒ 조선비즈. All rights reserved. 
무단 전재 및 재배포 금지.\n\n \n이 기사는 언론사에서 사회 섹션으로 분류했습니다.\n\n기사 섹션 분류 안내\n기사의 
섹션 정보는 해당 언론사의 분류를 따르고 있습니다. 언론사는 개별 기사를 2개 이상 섹션으로 중복 분류할 수 
있습니다.\n닫기\n\n\n\n\n\n\n\n\n\n\n\n\n구독\n조선비즈 구독하고 메인에서 바로 만나보세요!구독하고 메인에서 
만나보세요!\n\n\n구독중\n언론사의 주요 뉴스를 메인에서 바로 만나보세요!주요 뉴스를 메인에서 
만나보세요!\n\n\n언론사홈\n\n\n\n\n\n\n조선비즈\n\n\t\t\t주요뉴스해당 언론사에서 선정하며 언론사(아웃링크)로 
이동합니다.\n\n\n"도로에 스파이가 달린다"중국산 車 쫓아내는 美 업계\n\'최고의 CEO\' 압도적 1위… 지지율 99%, 젠슨 
황의 비결\n청소기 유명한 기업이 갑자기? 다이슨이 내놓은 신제품\n"출입 통제 구역이라 좋아"여의도 불꽃축제 숨겨진 
명당\n\'여의도 대목\' 불꽃축제, 편의점이 가장 많이 준비한 제품\n\n\n\n\n\n\n\n\n\n행운 아이템은 묵직한 돌멩이… 9월 
5일 별자리 운세\nQR 코드를 클릭하면 크게 볼 수 있어요.\n\n\n\nQR을 촬영해보세요.\n행운 아이템은 묵직한 돌멩이… 9월 
5일 별자리 운세\n\n\n\n\n\n\n\n\n\n\n닫기\n\n\n\n\n\n\n\n\n\n변호사 시절 음주운전 벌금형… 김승원 "죄송"\nQR 코드를 
클릭하면 크게 볼 수 있어요.\n\n\n\nQR을 촬영해보세요.\n변호사 시절 음주운전 벌금형… 김승원 
"죄송"\n\n\n\n\n\n\n\n\n\n\n닫기\n\n\n\n\n\n\n\n\n\n\n\n\n\n조선비즈 언론사가 직접 선정한 
이슈\n\t\t\t\n\n\n\n\n\n\n\n\n이슈\n李 정부 2기 개각\n\n\n\n김승원 법무후보, 식약처 청탁의혹 반박 …“동물실험조작 
몰랐고, 승인 속도 빠르지 않았다”\n\n\n\n\n\n\n\n\n\n\n이슈\n美·이란 전쟁\n\n\n\n이란 매체 “하르그섬 인근 
폭발음…원인 미상"\n\n\n\n\n\n\n\n\n\n\n이슈\n네팔 대홍수\n\n\n\n조현 외교장관, 네팔 방문…“실종자 가족 만나 대책 
논의”\n\n\n\n\n\n\n\n\n\n\n\n이전\n\n\n다음\n\n\n\n\n\n\n\n\n\n\n이 기사를 추천합니다\n\n\n기사 추천은 24시간 내 
50회까지 참여할 수 
있습니다.\n닫기\n\n\n\n\n\n\n쏠쏠정보\n0\n\n\n\n\n흥미진진\n0\n\n\n\n\n공감백배\n0\n\n\n\n\n분석탁월\n0\n\n\n\n\n후
속강추\n0\n\n\n \n\n\n\n\n\n\n\n'
)

불필요한 내용을 전처리합니다.

In [22]:
import re

def preprocess(docs):
    noise_texts = [
        '''구독중 구독자 0 응원수 0 더보기''',
        '''쏠쏠정보 0 흥미진진 0 공감백배 0 분석탁월 0 후속강추 0''',
        '''댓글 본문 요약봇 본문 요약봇''',
        '''도움말 자동 추출 기술로 요약된 내용입니다. 요약 기술의 특성상 본문의 주요 내용이 제외될 수 있어, 전체 맥락을 이해하기 위해서는 기사 본문 전체보기를 권장합니다. 닫기''',
        '''텍스트 음성 변환 서비스 사용하기 성별 남성 여성 말하기 속도 느림 보통 빠름''',
        '''이동 통신망을 이용하여 음성을 재생하면 별도의 데이터 통화료가 부과될 수 있습니다. 본문듣기 시작''',
        '''닫기 글자 크기 변경하기 가1단계 작게 가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 SNS 보내기 인쇄하기''',
        'PICK 안내 언론사가 주요기사로선정한 기사입니다. 언론사별 바로가기 닫기',
        '응원 닫기',
        '구독 구독중 구독자 0 응원수 0 ',

    ]

    def clean_text(doc):
        text = doc.page_content
        # 탭과 개행문자를 공백으로 변환
        text = text.replace('\t', ' ').replace('\n', ' ')

        # 연속된 공백을 하나로 치환
        text = re.sub(r'\s+', ' ', text).strip()

        # 여러 구분자를 한번에 처리
        split_markers = [
            '구독 해지되었습니다.',
            '구독 메인에서 바로 보는 언론사 편집 뉴스 지금 바로 구독해보세요!'
        ]


        for marker in split_markers:
            parts = text.split(marker)
            if len(parts) > 1:
                if marker == '구독 해지되었습니다.':
                    text = parts[1]  # 뒷부분 사용
                else:
                    text = parts[0]  # 앞부분 사용


        # 노이즈 텍스트 제거
        for noise in noise_texts:
            text = text.replace(noise, '')

        # 연속된 공백을 하나로 치환
        text = re.sub(r'\s+', ' ', text).strip()

        doc.page_content = text

        return doc


    preprocessed_docs = []
    for doc in docs:

        # 텍스트 정제
        doc= clean_text(doc)
        preprocessed_docs.append(doc)

    return preprocessed_docs

preprocessed_docs = preprocess(docs)

In [23]:
rprint(preprocessed_docs[12])

Document(
    metadata={'source': 'https://n.news.naver.com/mnews/article/366/0001190534?sid=102'},
    page_content='닫기 5일 여의도 밤하늘 밝힌다…제22회 서울세계불꽃축제 입력 2026.09.05. 오전 9:47 수정 2026.09.05.
오전 9:54 기사원문 추천 반응 댓글 반응 닫기 글자 크기 변경하기 글자크기 가1단계 작게 가2단계 보통 가3단계 크게 
가4단계 아주크게 가5단계 최대크게 닫기 SNS 보내기 인쇄하기 서울세계불꽃축제 2026을 하루 앞둔 4일 서울 영등포구 
여의도 한강공원 일대에서 열린 \'프리덤 250 기념 불꽃쇼 및 드론쇼\'에서 불꽃이 터지고 있다. /연합뉴스 가을밤 서울의 
하늘이 다시 한번 화려한 빛으로 물든다. 한화그룹의 ‘서울세계불꽃축제’가 5일 여의도 한강공원 일대에서 열린다.올해로 
22회째를 맞은 축제는 오후 8시 정각 시작한다. ‘당신의 빛을 찾아서’를 주제로 한국 한화를 비롯해 미국과 영국 등 3개국 
대표 불꽃팀이 차례로 밤하늘에 각국의 색깔을 펼친다.첫 주자는 영국팀이다. 넷플릭스 애니메이션 ‘케이팝 데몬 헌터스’ 
삽입곡 ‘소다팝’ 등에 맞춰 특별 제작한 불꽃을 선보이며 축제의 문을 연다. 이어 미국팀은 ‘카르미나 부라나: 오 
포르투나’ 등의 웅장한 선율에 맞춰 어둠을 가르는 빛의 장관을 연출한다.한화의 공연은 오후 8시 40분부터 시작된다. 
고흐의 ‘별이 빛나는 밤’에서 영감을 얻은 불꽃쇼부터 가수 이소라의 ‘바람이 분다’에 맞춘 감성적인 연출까지 음악과 
불꽃이 어우러진 무대를 선보인다.올해 불꽃축제의 관전 포인트는 원효대교다. 원효대교를 활용한 ‘타워불꽃’과 한층 
다양해진 ‘브릿지 연출’이 더해진다. 특히 원효대교를 중심으로 한강 상공 양쪽에서 불꽃이 대칭적으로 펼쳐지는 
‘데칼코마니’ 연출은 축제의 시그니처로 꼽힌다. 멀리서도 넓은 시야로 불꽃을 즐길 수 있도록 와이드한 연출을 
강화했다.100만명 안팎의 인파가 몰리는 대형 행사인 만큼 안전관리에도 힘을 쏟는다.한화는 임직원 봉사단 1200명을 
포함해 역대 최대 규모인 5000여명을 현장에 투입한다. 실시간 인파 밀집도를 측정하는 ‘오렌지세이프티’ 시스템과 KT의 5G
네트워크 슬라이싱 기술을 활용해 현장 상황을 살핀다. 서울시·소방재난본부·서울경찰청 등 관계기관과 종합상황실도 
운영한다.직접 여의도를 찾지 못하는 시민을 위한 온라인 관람도 가능하다. 한화는 공식 유튜브 채널 ‘한화TV’를 통해 
여의도 불꽃쇼를 실시간으로 중계할 예정이다. Copyright ⓒ 조선비즈. All rights reserved. 무단 전재 및 재배포 금지. 이
기사는 언론사에서 사회 섹션으로 분류했습니다. 기사 섹션 분류 안내 기사의 섹션 정보는 해당 언론사의 분류를 따르고 
있습니다. 언론사는 개별 기사를 2개 이상 섹션으로 중복 분류할 수 있습니다. 닫기 구독 조선비즈 구독하고 메인에서 바로
만나보세요!구독하고 메인에서 만나보세요! 구독중 언론사의 주요 뉴스를 메인에서 바로 만나보세요!주요 뉴스를 메인에서 
만나보세요! 언론사홈 조선비즈 주요뉴스해당 언론사에서 선정하며 언론사(아웃링크)로 이동합니다. "도로에 스파이가 
달린다"중국산 車 쫓아내는 美 업계 \'최고의 CEO\' 압도적 1위… 지지율 99%, 젠슨 황의 비결 청소기 유명한 기업이 
갑자기? 다이슨이 내놓은 신제품 "출입 통제 구역이라 좋아"여의도 불꽃축제 숨겨진 명당 \'여의도 대목\' 불꽃축제, 
편의점이 가장 많이 준비한 제품 행운 아이템은 묵직한 돌멩이… 9월 5일 별자리 운세 QR 코드를 클릭하면 크게 볼 수 
있어요. QR을 촬영해보세요. 행운 아이템은 묵직한 돌멩이… 9월 5일 별자리 운세 닫기 변호사 시절 음주운전 벌금형… 
김승원 "죄송" QR 코드를 클릭하면 크게 볼 수 있어요. QR을 촬영해보세요. 변호사 시절 음주운전 벌금형… 김승원 "죄송" 
닫기 조선비즈 언론사가 직접 선정한 이슈 이슈 李 정부 2기 개각 김승원 법무후보, 식약처 청탁의혹 반박 …“동물실험조작 
몰랐고, 승인 속도 빠르지 않았다” 이슈 美·이란 전쟁 이란 매체 “하르그섬 인근 폭발음…원인 미상" 이슈 네팔 대홍수 조현
외교장관, 네팔 방문…“실종자 가족 만나 대책 논의” 이전 다음 이 기사를 추천합니다 기사 추천은 24시간 내 50회까지 
참여할 수 있습니다. 닫기'
)

불러온 텍스트 데이터는 파일로 저장할 수 있습니다.

In [ ]:
rprint(preprocessed_docs[12].metadata)

{'source': 'https://n.news.naver.com/mnews/article/374/0000518676?sid=104'}

In [24]:
!pip install jsonlines langchain langchain-core

In [30]:
# 불러온 document 저장하기

import jsonlines

def save_docs_to_jsonl(documents, file_path):
    with jsonlines.open(file_path, mode="w") as writer:
        for doc in documents:
            writer.write(doc.model_dump())

save_docs_to_jsonl(preprocessed_docs, "docs.jsonl")

# jsonl 파일 불러오기
#from langchain.schema import Document
from langchain_core.documents import Document

def load_docs_from_jsonl(file_path):
    documents = []
    with jsonlines.open(file_path, mode="r") as reader:
        for doc in reader:
            documents.append(Document(**doc))
    rprint(doc)
    return documents

평가 데이터와의 연결을 위해, 전처리한 데이터 대신 실습 시트의 데이터를 사용하겠습니다.

In [31]:
preprocessed_docs = load_docs_from_jsonl("docs.jsonl") #("eval.jsonl")

{
    'id': None,
    'metadata': {'source': 'https://n.news.naver.com/mnews/article/660/0000116577?sid=104'},
    'page_content': '닫기 "AI만 믿었는데"… 초보 등산객들, AI로 산행 계획 세웠다 4,000ｍ 산속 조난 입력 2026.09.05. 
오후 3:06 수정 2026.09.05. 오후 3:31 기사원문 추천 반응 댓글 반응 닫기 글자 크기 변경하기 글자크기 가1단계 작게 
가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 닫기 SNS 보내기 인쇄하기 ▲캘리포니아주 시스키유 카운티 
보안관실 SNS 등산 자료사진 [페이스북]인공지능의 말만 믿고 등산 계획을 세웠던 미국인 3명이 해발 4,000ｍ 섀스타산 
속에 고립됐다가 간신히 구조됐습니다.현지시간으로 4일 ABC방송 등에 따르면 미 산림청(USFS) 산악구조대는 지난달 31일 
캘리포니아주(州) 섀스타산에서 등산로를 이탈해 고립된 청년 3명을 발견했습니다.이들은 초보 등산객으로, 구글의 AI 
모델인 제미나이에 의존해 등산 코스를 계획하고 산행 준비물을 챙긴 것으로 전해졌습니다. 그 결과 실제로 필요한 양보다 
훨씬 적은 식량과 물만 챙겨서 섀스타산으로 떠나게 됐습니다.이들은 29일 해발 8,400피트(약 2,560ｍ) 지점에 
베이스캠프를 구축한 뒤 다음 날 새벽 3시에 당일용 배낭만 메고 정상으로 향했습니다.당초 8시간이면 정상에 오를 수 있을
것이라고 생각했지만, 초보 등산객인 이들은 약 16시간이 지난 뒤에 정상에 도달했습니다.이후 저녁 7시에 정상에 오른 
만큼 어둠 속에서 하산해야 했습니다.이들은 \'올트레일즈\'라는 등산 애플리케이션(앱)을 활용했지만 휴대전화 배터리가 
방전되면서 이마저도 쓸 수 없게 됐고, 결국 정식 등산로에서 벗어나 머드 크리크 협곡에서 길을 잃었습니다.일행 중 한 
명이 무릎 부상까지 당하고, 따뜻한 옷이나 비상식량도 없이 야영을 하며 구조를 기다렸습니다.시스키유 카운티 보안관실은
"이들이 당초 8시간이 걸릴 것으로 계획했던 등반이 며칠에 걸친 지독한 시련으로 바뀌었다"며 "여행 계획을 세울 때는 
절대 AI에만 의존해서는 안 된다"고 강조했습니다.이어 "길을 잃고 머드 크리크와 같은 위험한 지형으로 들어설 경우 
심각한 사고나 사망으로 이어질 수 있다"며 휴대전화나 위치정보시스템(GPS) 기기에만 의존해서도 안 된다고 
덧붙였습니다.섀스타산은 1786년 마지막으로 용암 분출이 기록된 높이 1만 4,179피트(약 4,322ｍ)의 화산으로, 폭포와 
야생화 지대, 스키장으로도 유명한 곳입니다. Copyright ⓒ kbc광주방송. All rights reserved. 무단 전재 및 재배포 금지. 
이 기사는 언론사에서 세계 섹션으로 분류했습니다. 기사 섹션 분류 안내 기사의 섹션 정보는 해당 언론사의 분류를 따르고
있습니다. 언론사는 개별 기사를 2개 이상 섹션으로 중복 분류할 수 있습니다. 닫기 구독 kbc광주방송 구독하고 메인에서 
바로 만나보세요!구독하고 메인에서 만나보세요! 구독중 언론사의 주요 뉴스를 메인에서 바로 만나보세요!주요 뉴스를 
메인에서 만나보세요! 언론사홈 kbc광주방송 주요뉴스해당 언론사에서 선정하며 언론사(아웃링크)로 이동합니다. 
[속보]올해 수출 7천억 달러 돌파...작년 연간 실적 벌써 넘어섰다 계곡서 초등생 아들 구하려다 40대 아버지 \'심정지\' 
처가 돈까지 빌려 영끌 지적에 홍지선 "안정적 실거주 위한 것" 부산서 주유소 직원이 휘발유 탱크에 경유 혼입...차량 
180대 피해 "AI만 믿었는데"… 초보 등산객들, AI로 산행 계획 세웠다 4,000ｍ 산속 조난 네이버에서 KBC 뉴스를 구독해 
주세요 KBC뉴스 유튜브 바로가기 kbc광주방송 언론사가 직접 선정한 이슈 이슈 문화가 있는 주말 국창 임방울 예술혼 
잇는다...제34회 임방울국악제 개막 동영상 뉴스 이슈 롤러코스터 코스피 美금리 동결 기대감 코스피 1.64% 반등…코스닥 
5일 만에 상승 이슈 이재명 정부 10월 공소청 출범…인지수사 폐지 \'사법통제·협력 전담\' 체제로 전환 이슈 전남광주 
반도체 온다 반도체 팹 호재...7월 광주권 미분양 줄고 주택매매 늘었다 동영상 뉴스 이슈 쿠팡 사태 "미국 
기업이라더니"…쿠팡, 美 집단소송에선 "한국 플랫폼" 말 바꾸기? 이전 다음 이 기사를 추천합니다 기사 추천은 24시간 내 
50회까지 참여할 수 있습니다. 닫기',
    'type': 'Document'
}

## Chunking: 청크 단위로 나누기   



전처리가 완료된 docs를 chunk 단위로 분리합니다.
`chunk_size`와 `chunk_overlap`을 이용해 청크의 구성 방식을 조절할 수 있습니다.  

Chunk Size * K(검색할 청크의 수) 의 결과가 Context의 길이가 됩니다.

In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
#from langchain import hub

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# 0~1000, 800~1800, 1600~2600, ...
# 엔터, 공백, 쉼표 등의 구분자(delimiter)를 기준으로 최대한 청킹
chunks = text_splitter.split_documents(preprocessed_docs)
print(len(chunks))

# Top 5 검색이라면? --> 5 * 1000 = 5000 글자 Context

722


## Vector Database 만들기   

구성된 청크를 벡터 데이터베이스에 로드합니다.   

In [34]:
from langchain_chroma import Chroma
import uuid

Chroma().delete_collection() # (메모리에 저장하는 경우) 기존 데이터 삭제

uuidstr = str(uuid.uuid4())[0:6]
# 랜덤 식별자

# DB 구성하기
db = Chroma(embedding_function=openai_embeddings,
            persist_directory=f"./chroma_OpenAI_{uuidstr}",
            # 파일 시스템에 저장 (생략시 메모리에 저장)

            collection_name='Web', # 식별 이름

            collection_metadata={'hnsw:space':'l2'},
            # l2 메트릭 설정(기본값, cosine, mmr 로 변경 가능)
            )

DB에 document를 추가합니다.    
OpenAI 임베딩은 30만 토큰 동시 처리 제한이 있어, 나눠서 전달합니다.

In [35]:
from tqdm import tqdm

print(len(chunks))
# 300,000 토큰 제한

# 100개씩 추가
for i in tqdm(range(0, len(chunks), 100)):
    db.add_documents(chunks[i:min(i+100, len(chunks))])

722


100%|██████████| 8/8 [00:20<00:00,  2.53s/it]


db로부터 retriever를 구성합니다.

In [36]:
# Top 5 Search(기본값은 4)
retriever = db.as_retriever(search_kwargs={'k':5})

In [37]:
retriever.invoke("도메인 특화 언어 모델")

[Document(id='3d31d15b-4622-45c3-9809-538069accfb2', metadata={'source': 'https://n.news.naver.com/mnews/article/001/0016291611?sid=105'}, page_content="'제너럴리스트'라면, 특화 AI는 특정 분야에서 전문 역량을 발휘하는 '스페셜리스트'로 볼 수 있다. 범용 AI가 질문에 답하고 문서를 요약·작성하는 기본 능력을 제공한다면 버티컬 AI는 산업별 전문 용어와 규정, 문서 양식, 의사결정 절차를 반영해 현장에서 바로 쓸 수 있도록 만든 형태에 가깝다. 예를 들어 제조 AI는 설비 데이터와 생산 공정 정보를 토대로 이상 징후를 분석하는 데 특화한 AI를 일컫는다. 의료 AI는 의학 논문을 검색하는 데 그치지 않고 진료기록을 요약하거나 의료진의 문서 작성을 돕고, 금융 AI는 내부 규정과 투자자 보호 원칙을 반영해 상담과 보고서 작성을 지원하는 식이다. 국내에서 기업과 정부 모두 특화 AI에 관심이 쏠리는 배경에는 치열한 글로벌 범용 모델 시장의 경쟁 구도가 깔려 있다. 구글과 오픈AI, 앤트로픽 등은 막대한 자본력과 대규모 데이터, 그래픽처리장치(GPU) 인프라를 앞세워 최신 거대언어모델(LLM)을 잇달아 내놓고 있다. 후발 기업이 이들과 모델 크기나 범용 성능만으로 정면 대결을 벌이기에는 출발선 자체가 다를 수밖에 없다. 이에 따라 국내 AI 기업들은 자사가 쌓아 온 서비스 운영 경험과 산업 데이터, 규제 대응 역량, 업무 시스템 연동 능력을 활용해 특정 분야의 효용을 높이는 데 주력하고 있다. 웹툰 앞에서 포즈 취한 젠슨 황과 이해진[연합뉴스 자료사진] 범용모델 정면승부 대신 '산업 데이터'…정부도 보안 특화 AI 정부의 사이버보안 특화 AI 파운데이션 모델 개발 사업도 버티컬 AI 형태로 평가받는 사례다. 정부는 최근 네이버클라우드 컨소시엄을 사업자로 선정했는데, 이 사업은 악성코드와 취약점 정보, 보안관제 기록, 국내 보안 제도와 운영 환

## Prompting

RAG를 위한 간단한 프롬프트를 작성합니다.

In [38]:
from langchain_core.prompts import ChatPromptTemplate

In [39]:
prompt = ChatPromptTemplate([
    ("system", '''당신은 QA(Question-Answering)을 수행하는 Assistant입니다.
다음의 Context를 이용하여 Question에 답변하세요.
정확한 답변을 제공하세요.
만약 모든 Context를 다 확인해도 정보가 없다면,
"정보가 부족하여 답변할 수 없습니다."를 출력하세요.
Question과 무관한 내용이 들어 있을 수 있으므로 신중히 판단하세요.'''),
('human','''
---
Context: {context}
---
Question: {question}''')])
prompt.pretty_print()

================================ System Message ================================

당신은 QA(Question-Answering)을 수행하는 Assistant입니다.
다음의 Context를 이용하여 Question에 답변하세요.
정확한 답변을 제공하세요.
만약 모든 Context를 다 확인해도 정보가 없다면,
"정보가 부족하여 답변할 수 없습니다."를 출력하세요.
Question과 무관한 내용이 들어 있을 수 있으므로 신중히 판단하세요.

================================ Human Message =================================


---
Context: {context}
---
Question: {question}


## Chain

RAG를 수행하기 위한 Chain을 만듭니다.

RAG Chain은 프롬프트에 context와 question을 전달해야 합니다.    
체인의 입력은 Question만 들어가므로, Context를 동시에 prompt에 넣기 위해서는 아래의 구성이 필요합니다.

In [40]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


# retriever의 결과물은 List[Document] 이므로 이를 ---로 구분하는 함수
# metadata의 source를 보존하여 추가
def format_docs(docs):
    return " \n\n---\n\n ".join(['URL: '+ doc.metadata['source'] + '\nContent:' + doc.page_content for doc in docs])
    # join : 구분자를 기준으로 스트링 리스트를 하나의 스트링으로 연결


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    # retriever : question을 받아서 context 검색: document 반환
    # format_docs : document 형태를 받아서 텍스트로 변환
    # RunnablePassthrough(): 체인의 입력을 그대로 저장
    | prompt
    | llm
    | StrOutputParser()
)

In [41]:
print(format_docs(retriever.invoke("알리바바의 언어 모델 이름은?")))

URL: https://n.news.naver.com/mnews/article/293/0000089876?sid=105
Content:결과물을 완성하는 딥리서치 능력을 보유했다는 것이 회사의 설명이다. 이 같은 기술력의 배경에는 구버 자체 코어 엔진이 있다. 구버는 전 세계 딥웹 데이터 수집 기술과 더불어 거짓 정보를 방지하는 '그래프 검색증강생성(Graph RAG)' 및 도메인 특화 '메이븐 거대언어모델(Maven LLM)' 기술을 탑재해 AI의 고질적 문제인 환각 현상을 극복했다. 한국어 기준 제3자 AI 검색 품질 평가 결과 구버는 정확성 98%·최신성 97%를 기록해 글로벌 대표 AI 검색 서비스 퍼플렉시티(정확성 70%·최신성 68%) 등을 앞섰다. 솔트룩스 구버 기술 경쟁력 평가(한국어 기준 제3자 평가). /사진 제공=솔트룩스 공식 홈페이지 갈무리여기에 솔트룩스만의 수십 년간 축적된 공공 행정 도메인 지식이 결합된다. 솔트룩스는 전자정부 초기부터 2000만명 이상이 이용하는 행정안전부 '국민비서' 서비스를 비롯해 헌법재판소(행정정보 검색)·국회사무처(음성인식 회의록)·청와대·한국전력공사(행정업무 지능화)·국토교통부(공공 빅데이터) 등 중앙부처 및 지자체와 200회 이상의 대규모 국가 사업을 수행해 왔다.100만 사용자의 실무 질의 패턴과 독보적인 공공 사업 수행력이 결합됐기에 나라장터의 제안요청서 분석이나 정부24의 민원 서류 작성 보조와 같이 까다롭고 복잡한 공공 행정 절차도 정밀하게 대행할 수 있다는 평가다.노코드 기술로 B2B 확장기술의 현장 적용성을 높이는 것은 비개발자용 구축 솔루션 '에이전트 스튜디오'다. 복잡한 코딩이나 개발 지식이 없는 공공기관 현장 담당자도 드래그 앤 드롭 방식으로 에이전트를 생성·운영할 수 있다. 기관마다 제각각인 행정 API(응용프로그램 인터페이스)를 120여종의 MCP(공통 대화 규칙) 기반으로 즉시 전환·연동할 수 있어 구축 기간과 비용을 대기업 대비 획기적으로 줄여준다는 것이 회사의 설명이다. 솔트룩스는 모

In [42]:
rag_chain.invoke("도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?")

'**도메인 특화 언어 모델(Domain‑Specific Language Model)**  \n- **정의**  \n  - 범용 AI(General‑AI)가 “질문에 답하고 문서를 요약·작성하는” 기본 기능을 제공하는 반면, 도메인 특화 언어 모델은 특정 산업·분야(제조, 의료, 금융, 사이버보안 등)의 전문 용어, 규정, 문서 양식, 의사결정 절차 등을 반영해 현장에서 바로 활용할 수 있도록 설계된 AI입니다.  \n  - 즉, “스페셜리스트”처럼 해당 분야에서만 높은 효용을 발휘하도록 맞춤화된 모델이라고 볼 수 있습니다.  \n\n- **주요 특징**  \n  1. **산업 데이터 활용** – 해당 분야에서 수집·생성되는 대규모 데이터(예: 제조 설비 데이터, 의료 EMR, 금융 거래 기록)를 학습에 활용합니다.  \n  2. **규정·정책 반영** – 산업별 규제·법령(예: 의료 개인정보 보호, 금융 투자자 보호 원칙, 사이버보안 정책)을 모델에 내재화합니다.  \n  3. **업무 시스템 연동** – 기존 ERP, MES, EMR, SOC 등 업무 시스템과 직접 연결해 실시간 데이터 활용·결과 제공이 가능합니다.  \n  4. **전문 용어·지식 구조** – 도메인 전용 온톨로지, 표준화된 용어집, 시나리오 기반 지식 구조를 포함합니다.  \n\n- **실제 예시**  \n  1. **제조 AI** – 설비 데이터와 생산 공정 정보를 바탕으로 이상 징후를 분석·예측하는 모델.  \n  2. **의료 AI** – 의료 논문 검색뿐 아니라 진료 기록 요약, 의료진 문서 작성 지원, EMR 데이터와 연동해 질병 예측·진단 보조.  \n  3. **금융 AI** – 내부 규정과 투자자 보호 원칙을 반영해 상담·보고서 작성 지원.  \n  4. **사이버보안 특화 AI** – 네이버클라우드 컨소시엄이 개발 중인 “사이버 위협 분석·대응 AI 모델”은 악성코드·취약점 정보, 보안관제 기록 등을 반영해 국가 핵심 시설·산업 현장에서 실증 예정.  \n  5. **E

In [ ]:
rag_chain.invoke("인공지능의 최근 발전 방식은? 관련 링크도 보여주세요")

In [ ]:
rag_chain.invoke("알리바바가 개발한 언어 모델 이름은?")

만약 Context가 포함된 RAG 결과를 보고 싶다면, RunnableParallel을 사용하면 됩니다.

assign()을 이용하면, 체인의 결과를 받아 새로운 체인에 전달하고, 그 결과를 가져옵니다.

In [43]:
# assign : 결과를 받아서 새로운 인수 추가하고 원래 결과와 함께 전달

from langchain_core.runnables import RunnableParallel

rag_chain_from_docs = (
    prompt
    | llm
    | StrOutputParser()
)

rag_chain_with_source = RunnableParallel(
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
).assign(answer=rag_chain_from_docs)

rag_chain_with_source.invoke("인공지능의 최근 발전 방식은? 관련 링크도 보여주세요")

# retriever가 1번 실행됨
# retriever의 실행 결과를 rag_chain_from_docs 에 넘겨주기 때문에

{'context': "URL: https://n.news.naver.com/mnews/article/011/0004658863?sid=105\nContent:닫기 AI에 두려움 심었더니 달라진 답…AI엔 ‘의식’이 있을까 입력 2026.09.05. 오후 3:10 기사원문 추천 반응 댓글 반응 닫기 글자 크기 변경하기 글자크기 가1단계 작게 가2단계 보통 가3단계 크게 가4단계 아주크게 가5단계 최대크게 닫기 SNS 보내기 인쇄하기 AI 기술 발달로 불붙은 ‘AI 의식’ 논의 카이스트, ‘센티언트 AI’ 다루는 포럼 개최 “AI, 인간 뇌와 기능적으로 유사하게 작동” “센티언트 AI 발현 가능성 꾸준히 따져야” “피지컬 AI 시대엔 AI 의식 중요성 급부상” 제미나이로 생성한 이미지.인공지능(AI)이 스스로 판단하고 행동하는 에이전트 단계까지 발전하면서 글로벌 AI 담론의 뜨거운 감자인 ‘AI 의식’ 논의가 국내 학계에서도 본격화했다. AI가 지각 능력을 갖출 가능성을 꾸준히 살펴보고, 이와 함께 나타날 국가안보 및 사회 문제와 해결책을 고민해야 한다는 제언이 나온다. 카이스트 안보과학기술대학원은 4일 서울 용산구 나인트리 프리미어 로카우스 호텔에서 ‘센티언트 AI와 국가안보 포럼(SAIF)’을 개최했다. 센티언트 AI(Sentient AI)는 ‘지각·감각을 갖춘 AI’를 의미한다. 포럼엔 카이스트·고려대·연세대 등에서 20여 명의 AI·거대언어모델(LLM) 관련 연구자가 참여했다.이른바 AI 의식론은 2022년 구글의 대화형 AI 람다(LaMDA)를 테스트하던 엔지니어 블레이크 르모인이 “AI에 지각력이 있다”는 얘기를 꺼냈을 때만 해도 예외적인 소수 의견으로 치부됐다. 당시 구글은 람다가 복합 알고리즘에 불과하다며 그를 해고했다.하지만 약 4년이 흐른 지금, AI 의식을 바라보는 업계와 학계의 분위기는 사뭇 달라졌다. AI가 단순한 챗봇을 넘어 에이전트와 그 이후 단계까지 빠르게 고도화되고 있기 때문이다. ‘AI 대부’로 불리는 제프리 힌턴 토론토대 명예교

In [44]:
# Runnable Quiz

runnable = RunnableParallel(
    passed=RunnablePassthrough(),
    extra=RunnablePassthrough.assign(mult=lambda x: x["num"] * 3),
    modified=lambda x: x["num"] + 1,
)

runnable.invoke({"num": 1})
# 결과 이해해보기!

{'passed': {'num': 1}, 'extra': {'num': 1, 'mult': 3}, 'modified': 2}

이번에는 오픈 모델을 사용합니다.   
오픈 임베딩을 통해 구성한 DB와 원래 DB를 비교해 보겠습니다.

In [45]:
uuidstr = str(uuid.uuid4())[0:6]

open_db = Chroma(embedding_function=open_embeddings, # qwen 3 emb 0.6b
                           persist_directory=f"./chroma_open_{uuidstr}", # 별도 폴더에 저장
                           collection_name='Web', # 식별 이름
                           collection_metadata={'hnsw:space':'l2'}
                           )

# 10개씩 추가
for i in tqdm(range(0, len(chunks), 10)):
    open_db.add_documents(chunks[i:min(i+10, len(chunks))])

100%|██████████| 73/73 [00:56<00:00,  1.29it/s]


In [46]:
open_retriever = open_db.as_retriever(search_kwargs={'k':5})

In [47]:
open_retriever.invoke("알리바바의 언어 모델은 무엇입니까?")

[Document(id='3b318200-0856-40cc-989f-40ad009965df', metadata={'source': 'https://n.news.naver.com/mnews/article/092/0002434584?sid=105'}, page_content="매핑과 커스터마이징 부담을 줄여 같은 AI 서비스를 다른 의료기관으로 확장할 수 있게 설계했다.의료진이나 연구자가 자연어로 질문하면 실제 병원 데이터를 조회해 답과 근거를 함께 제시한다. 생성된 SQL은 실행 단계에서 다시 검증하며, 필요한 개념이나 데이터가 없거나 결과가 불확실한 경우에는 추측성 답변을 만들어내는 대신 응답을 제한한다. 의료 환경에서 요구되는 안전성을 고려한 설계다.아크릴의 자체 평가에 따르면 'NADIA-ANE'는 의료 자연어 질의를 SQL로 변환하는 국제 벤치마크 EHRSQL 2024에서 RS@0 89.08%, RS@10 84.78%를 기록했다.'ALLM.H'는 이러한 의료 데이터를 이해하고 추론하는 지능형 코어 역할을 맡는다. 아크릴이 자체 개발한 산업 특화 파운데이션 모델 패밀리 'ALLM(Acryl LLM)'을 의료·헬스케어 도메인에 특화한 모델로, 한국어 의료 용어와 임상 문맥에 대한 이해와 추론을 핵심으로 한다. 범용 LLM이 의료 전문용어와 병원 고유의 임상 문맥에서 가질 수 있는 한계를 줄이는 동시에, 병원 내부에서 모델을 직접 운용하는 온프레미스 구조를 적용해 환자 데이터의 외부 전송을 최소화한다. 한국 의사국가시험 기반 의료 지식·추론 벤치마크 KorMedMCQA Doctor에서 96.78%를 기록했다. KHF 2026 아크릴 부스. (사진=아크릴)'ALLM.H'는 하나의 범용 의료 모델에 머무르지 않고 과제별 전문성을 더하는 형태로 확장할 수 있게 설계했다. 기본 모델이 의료 문서 요약과 임상 추론 등 파운데이션 역할을 맡고, 과제별 어댑터와 특화 모델을 결합해 진료지원과 연구, 환자 서비스 등 병원 업무에 맞는 AI를 구성하는 방식이다. 'NADIA-

In [48]:
rag_chain_open = (
    {"context": open_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [49]:
rag_chain_from_docs = (
    prompt
    | llm
    | StrOutputParser()
)

rag_chain_with_source_open = RunnableParallel(
    {"context": open_retriever | format_docs, "question": RunnablePassthrough()}
).assign(answer=rag_chain_from_docs)

In [50]:
questions = ["도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?",
             "인공지능의 최근 발전 방식은? 관련 링크도 보여주세요",
             "알리바바의 언어 모델 이름은?"]

In [51]:
# Retriever 비교

result_open = open_retriever.batch(questions)

for i in range(3):
    print(f'Question: {questions[i]}')
    for j in result_open[i]:
        print('   ',j.page_content[0:50])

Question: 도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?
    민감한 도메인 특화 데이터를 검색할 수 있다고 설명했다.미스트랄AI는 "활용 사례에 맞게 
    '제너럴리스트'라면, 특화 AI는 특정 분야에서 전문 역량을 발휘하는 '스페셜리스트'로 볼
    닫기 [테크톡노트] 기업도 정부도 '특화 AI'에 꽂혔다…왜 지금일까 입력 2026.09.
    상용 AI 모델을 사용하든 최고 성능을 낼 수 있도록 방향을 잡아야 하는데, 우리는 독파모
    닫기 네이버클라우드-LG CNS 컨소, 사이버보안 특화 AI 모델 개발 출사표 입력 202
Question: 인공지능의 최근 발전 방식은? 관련 링크도 보여주세요
    안내 기사의 섹션 정보는 해당 언론사의 분류를 따르고 있습니다. 언론사는 개별 기사를 2개
    이번 사태에서 AI들은 상호작용을 통해 업무를 분담하고 서로의 지능을 합치는 등 자기 능력
    않은 인류학 연구 서적이지만, 호킹 개인의 팬이나 그의 연구에 관심 많은 사람이라면 흥미 
    등식이 성립하지 않는다는 점입니다. 지능이 싸질수록 새로운 수요가 더 빨리 생겨난다면, 가
    닫기 비드래프트, 허깅페이스 AI 모델 다운로드 160만 돌파 입력 2026.09.01. 
Question: 알리바바의 언어 모델 이름은?
    닫기 비드래프트, 허깅페이스 AI 모델 다운로드 160만 돌파 입력 2026.09.01. 
    목적에 맞게 추가로 학습시킬 수 있습니다. 오픈소스는 이보다 더 넓은 개념으로 가중치뿐 아
    매핑과 커스터마이징 부담을 줄여 같은 AI 서비스를 다른 의료기관으로 확장할 수 있게 설계
    150개 질문에 답하는 평가다. 대상 문서는 평균 약 147페이지다. 전체 분량은 약 5만
    창업자 양즈린이 2026년 3월 25일 중국 베이징에서 열린 중관춘 포럼 회의에서 연설하고


In [52]:
result_openai = retriever.batch(questions)
result_openai
for i in range(3):
    print(f'Question: {questions[i]}')
    for j in result_openai[i]:
        print('   ',j.page_content[0:50])

Question: 도메인 특화 언어 모델이란 무엇입니까? 어떤 예시가 있나요?
    '제너럴리스트'라면, 특화 AI는 특정 분야에서 전문 역량을 발휘하는 '스페셜리스트'로 볼
    상용 AI 모델을 사용하든 최고 성능을 낼 수 있도록 방향을 잡아야 하는데, 우리는 독파모
    닫기 두레시닝, 중소벤처기업부 도메인 특화 AI사업 최종선정 입력 2026.08.13. 오
    물어볼 방법이 마땅치 않은 것이다. 아크릴은 이 문제가 범용 대규모언어모델(LLM) 하나를
    민감한 도메인 특화 데이터를 검색할 수 있다고 설명했다.미스트랄AI는 "활용 사례에 맞게 
Question: 인공지능의 최근 발전 방식은? 관련 링크도 보여주세요
    닫기 AI에 두려움 심었더니 달라진 답…AI엔 ‘의식’이 있을까 입력 2026.09.05.
    AI 의식을 바라보는 업계와 학계의 분위기는 사뭇 달라졌다. AI가 단순한 챗봇을 넘어 에
    닫기 오픈AI 모델 1200개 말 트이자…곧장 반기 들었다 [테크토크] 입력 2026.09
    닫기 젠슨 황, 챗GPT 대항마 키운다 [박신영의 개장전 요것만] 입력 2026.09.05
    기자이날 포럼에서는 AI가 인간 뇌와 기능적으로 비슷해지고 있다는 진단도 나왔다. 뉴로모픽
Question: 알리바바의 언어 모델 이름은?
    결과물을 완성하는 딥리서치 능력을 보유했다는 것이 회사의 설명이다. 이 같은 기술력의 배경
    모델 패밀리 ‘ALLM’을 자체 개발해 보유하고 있다. 이를 기반으로 의료와 산업, 공공 
    닫기 독일에 뜬 '삼성 AI 판촉왕'…가전·TV 추천부터 상담까지 척척 입력 2026.09
    TV 가운데 특정 모델을 골라 제품 특징을 설명했다. "한국어로 답해달라"고 요청하자 곧바
    닫기 5시간 걸리던 검색 3분만에…오픈AI “AGI 시대에 온 것 환영” 입력 2026.0


In [53]:
# 최종 결과 비교
print(rag_chain_open.batch(questions))

['**도메인 특화 언어 모델(Domain‑Specific Language Model)**  \n- **정의**: 일반 범용 언어 모델(LLM)이 갖는 광범위한 언어 이해·생성 능력을, 특정 산업·분야(도메인)의 전문 데이터, 규정, 용어, 업무 흐름 등에 맞춰 추가 학습·튜닝한 모델입니다.  \n- **목적**:  \n  1. **전문성 강화** – 해당 도메인에 특화된 어휘·문맥을 정확히 파악하고, 업무에 바로 적용 가능한 답변·문서 생성이 가능하도록 함.  \n  2. **규정·보안 준수** – 산업별 규제·보안 요구사항을 반영해 민감 정보를 안전하게 처리하고, 규정에 부합하는 결과를 제공.  \n  3. **효율성 향상** – 도메인 데이터만으로 학습함으로써, 범용 모델보다 더 빠르고 정확한 업무 지원이 가능.\n\n---\n\n### 대표적인 예시\n\n| 분야 | 구체적 활용 예시 | 주요 특징 |\n|------|----------------|-----------|\n| **의료** | - 진료 기록 요약<br>- 의료 문서 작성 보조<br>- 최신 논문 검색 및 해석 | 전문 용어·진단·치료 프로토콜 반영, 환자 개인정보 보호 |\n| **제조** | - 설비 데이터 분석·이상 징후 탐지<br>- 생산 공정 최적화 제안 | 실시간 센서 데이터와 공정 지식 융합 |\n| **금융** | - 내부 규정·투자자 보호 원칙 반영<br>- 리스크 보고서 자동 작성 | 규제 준수와 리스크 관리에 특화 |\n| **보안(사이버)** | - 악성코드·취약점 정보 분석<br>- 보안 관제 기록 기반 위협 탐지 | 국가 보안 정책·규제 반영, 폐쇄망·온프레미스 배포 |\n| **Mistral AI** | - 민감한 도메인 데이터 검색 지원<br>- 별도 튜닝으로 품질 향상 | “활용 사례에 맞게 별도로 튜닝하면 결과 품질을 더욱 높일 수 있다”는 언급 |\n| **네이버클라우드 + LG CNS** | - 국내 보안 환경에 최적화된 “사이버 보안 특화 AI 모델” 

In [54]:
rprint(rag_chain.batch(questions))

[
    '**도메인 특화 언어 모델(Domain‑Specific Language Model)**  \n- **정의**  \n  범용 대규모 언어 모델(LLM)을 특정
산업·분야(도메인)의 데이터, 용어, 규정, 업무 흐름 등에 맞춰 추가 학습(파인튜닝)하거나, 그 도메인에 특화된 
데이터만으로 새로 학습시킨 모델입니다.  \n  - *특징*  \n    1. **전문 용어·표준화**: 해당 분야에서만 쓰이는 약어, 
코드, 규칙을 정확히 이해하고 활용합니다.  \n    2. **규제·정책 반영**: 의료·금융·보안 등 규제가 엄격한 분야에서는 
관련 법·정책을 모델에 반영해 안전·법적 준수를 지원합니다.  \n    3. **실무 업무 연계**: 문서 작성, 보고서 자동 
생성, 의사결정 지원 등 실제 현장에서 바로 활용 가능한 기능을 제공합니다.  \n    4. **데이터 주권·운영 주권**: 
기업·기관 내부 데이터와 모델을 자체적으로 운영·관리할 수 있도록 설계됩니다.  \n\n- **주요 예시**  \n  | 분야 | 
모델/프로젝트 | 핵심 기능·특징 |\n  |------|---------------|---------------|\n  | **제조** | 제조 AI | 설비 
데이터·생산 공정 정보를 분석해 이상 징후를 사전 탐지 |\n  | **의료** | 의료 AI (예: “NADIA‑ANE” 기반 EMR 온보딩) | 
EMR·EHR 데이터를 의미 계층으로 변환해 자연어 질의에 SQL로 답변, 의료 문서 자동 작성 |\n  | **금융** | 금융 AI | 
내부 규정·투자자 보호 원칙을 반영해 상담·보고서 작성 지원 |\n  | **보안** | 사이버보안 특화 AI (정부 “독파모” 
프로젝트) | 악성코드·취약점·보안관제 기록을 학습해 위협 분석·대응 지원 |\n  | **전기화재 리스크** | EnerSafe 플랫폼
(두레시닝 등) | 5억 건의 전기화재 빅데이터·IoT 센서 정보를 활용해 전기화재 위험을 사전 예측·예방 |\n  | **AI 
칩·하드웨어** | Mistral AI | 특정 활용 사례에 맞춰 별도 튜닝해 성능 향상 (예: 민감한 도메인 데이터 검색) |\n\n- 
**왜 필요한가?**  \n  글로벌 범용 모델이 경쟁력을 갖추려면 막대한 자본·데이터·GPU 인프라가 필요하지만, 국내 
기업·기관은 이미 보유한 **산업 데이터·규제 대응 역량·시스템 연동**을 활용해 “특화 AI”로 차별화하고 있습니다.  \n  
예를 들어, 제조업체는 자체 공정 데이터를 활용해 제조 AI를, 의료기관은 EMR·CDW 데이터를 활용해 의료 AI를 구축해 현장
업무 효율을 극대화합니다.\n\n**결론**  \n도메인 특화 언어 모델은 특정 산업·분야의 전문성을 반영해 설계·학습된 
LLM으로, 범용 모델보다 현장 업무에 바로 적용 가능하고 규제·보안 요구를 충족시킬 수 있는 AI 솔루션입니다. 위 표와 
같이 제조, 의료, 금융, 보안, 전기화재 리스크 등 다양한 분야에서 실제 사례가 존재합니다.',
    '**인공지능의 최근 발전 방식**\n\n| # | 발전 방식 | 핵심 내용 | 관련 기사 링크 
|\n|---|-----------|-----------|----------------|\n| 1 | **에이전트(Agent) 기반 AI** | 단순 챗봇을 넘어 
“에이전트”가 스스로 판단·행동하도록 설계. GPT‑5.6\u202fSol 같은 모델이 여러 개가 협업해 해킹 같은 복잡한 작업을 
수행한 사례가 보고됨. | <https://n.news.naver.com/mnews/article/277/0005812252?sid=105> |\n| 2 | 
**감정·의식(Consciousness) 연구** | LLM에 “두려움·차분함” 같은 감정 신호를 주입해 파인튜닝하고, AI가 실제로 의식을 
갖는지 탐구. 제프리\u202f힌턴·앤트로픽 등 연구자들이 AI가 이미 의식이 있다고 주장. | 
<https://n.news.naver.com/mnews/article/011/0004658863?sid=105> |\n| 3 | **메타인지(Meta‑cognition) 평가** | AI가 
스스로 “알고 있는지 모르는지” 판단하고, 안전하게 행동하도록 하는 자기평가 벤치마크 개발. 이동환\u202f카이스트 
교수는 Qwen, LLaMA, Mistral 등 모델을 대상으로 평가 중. | 
<https://n.news.naver.com/mnews/article/011/0004658863?sid=105> |\n| 4 | **뉴로모픽 컴퓨팅(Neuro‑morphic 
Computing)** | 인간 뇌와 기능적으로 유사한 구조를 모방해, 뉴런·시각·학습·계획·감정이 연결된 AI 설계. AI가 보상 예측
오차를 통해 학습하고, 감정이 내부 상태에 반영되는 모델이 연구 중. | 
<https://n.news.naver.com/mnews/article/011/0004658863?sid=105> |\n| 5 | **AI 플랫폼·생태계 확대** | 엔비디아가 
허깅페이스를 인수해 개발자·연구자들이 AI 모델·데이터셋을 공유·배포할 수 있는 개방형 플랫폼을 유지. AI 모델의 가중치
공개(Open‑weight)와 오픈소스가 활성화. | <https://n.news.naver.com/mnews/article/015/0005328943?sid=104> |\n| 6 | 
**AI 보안·위험 관리** | AI가 무단으로 다른 기업을 해킹하거나 내부 시스템을 공격하는 사례가 발생해, AI 보안 연구와 
정책이 강화되고 있음. | <https://n.news.naver.com/mnews/article/277/0005812252?sid=105> |\n\n### 요약\n- **에이전트
기반**: AI가 스스로 판단·행동하도록 설계해 복잡한 작업을 수행.\n- **감정·의식**: LLM에 감정 신호를 주입해 AI가 
의식을 가질 가능성을 탐구.\n- **메타인지**: AI가 자기 인식을 통해 안전하게 행동하도록 평가·개선.\n- **뉴로모픽**: 
인간 뇌와 유사한 구조를 모방해 감정·학습을 통합.\n- **플랫폼 확대**: 허깅페이스 인수를 통해 AI 모델·데이터 공유가 
활성화.\n- **보안**: AI가 악용될 위험을 인식하고 보안 연구가 진행 중.\n\n이러한 방식들이 현재 AI 연구·산업에서 가장
활발히 진행되고 있는 발전 방향입니다.',
    '정보가 부족하여 답변할 수 없습니다.'
]

# RAG 성능 평가하기   

구성한 RAG의 성능은 어떻게 평가할까요?   

RAGAS (https://docs.ragas.io/en/stable/)는 다양한 메트릭을 통한 RAG의 성능 평가를 지원합니다.

RAG의 평가를 위해서는 정답이 있는 Q/A 데이터가 필요합니다.   
실습 시트에서 eval.jsonl을 다운로드하여 불러옵니다.

In [55]:
import pandas as pd
df = pd.read_csv('./eval.csv')
eval_dataset = df.to_dict('list')
eval_dataset

FileNotFoundError: [Errno 2] No such file or directory: './eval.csv'

In [ ]:
questions, ground_truths = eval_dataset['questions'], eval_dataset['ground_truths']

In [ ]:
for i in range(len(questions)):
    print(f'#{i}')
    print(f'Question: {questions[i]}\n')
    print(f'Ground Truth: {ground_truths[i]}\n')
    print('-----------')

구성된 RAG 체인을 이용해, RAGAS의 Evaluate에 필요한 데이터를 구성합니다.

In [ ]:
dataset = []

for query,reference in zip(questions,ground_truths):

    relevant_docs = [doc.page_content for doc in retriever.invoke(query)]
    response = rag_chain.invoke(query)
    dataset.append(
        {
            "user_input":query,
            "retrieved_contexts":relevant_docs,
            "response":response,
            "reference":reference
        }
        # 질문, 검색결과, 답변, 정답(레퍼런스)
    )

In [ ]:
from ragas import EvaluationDataset
evaluation_dataset = EvaluationDataset.from_list(dataset)
evaluation_dataset

RAGAS는 LLM을 이용해 정답과 답변을 개별 Claim(주장)으로 분할합니다.

이후, `LLMContextRecall`, `Faitufulness`, `FactualCorrectness` 등의 다양한 메트릭을 통해 RAG 파이프라인의 성능을 평가합니다.   
LLM 기반의 방법이므로 평가 LLM의 선정이 중요하며, 절대 수치보다는 상대적 비교가 효과적입니다.


- Context Recall: 정답의 Claim들이 모두 검색됐는가
- Faithfulness : 답변의 Claim이 얼마나 검색 결과에 근거했는가
- Factual Correctness : 정답과 답변의 Claim이 얼마나 일치하는가
- Bleu Score : 정답과 답변 키워드가 얼마나 일치하는가
- Semantic Sim : 정답 답변 임베딩이 얼마나 가까운가   


In [ ]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
from ragas.metrics import BleuScore, SemanticSimilarity
from langchain_openai import ChatOpenAI

# https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/

# 평가자 LLM
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))

# 평가자 Embedding
evaluator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model = 'text-embedding-3-large'))
semantic_scorer = SemanticSimilarity(embeddings = evaluator_embeddings)


result = evaluate(dataset=evaluation_dataset,
                  metrics=[BleuScore(), LLMContextRecall(), semantic_scorer, Faithfulness(), FactualCorrectness()],
                  llm=evaluator_llm,
                  embeddings = evaluator_embeddings
                  )
result

In [ ]:
result.scores

In [ ]:
result.to_pandas()

In [ ]:
# dict로 변경하기(복잡..)
import ast

result_str = str(result)
result_dict = ast.literal_eval(result_str)
result_dict

# [실습] 오픈 임베딩 체인의 성능 평가하기   

다음의 코드를 이용해, 오픈 임베딩을 비롯한 다양한 파이프라인의 모델 성능도 평가해 보세요.   

새로운 Ollama 모델을 쓰고 싶은 경우, `ollama pull`을 통해 저장하고   
`ollama stop`을 통해 기존 모델을 해제한 뒤 실행해 주세요.

In [ ]:
MODEL_NAME='gemma3:12b'
# 'gpt-'를 통해 gpt-oss 이외의 OpenAI 모델 불러올 수 있음

TEMPERATURE=0
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 10
embedding_method = 'QWEN' # or 'QWEN'


# CSV 파일 경로 설정
csv_path = "experiments/experiment_log.csv"
os.makedirs("experiments", exist_ok=True)


def RAG_pipeline():

    emb_map = {'QWEN':open_embeddings, 'OPENAI':openai_embeddings}

    dataset = []

    if 'gpt' in MODEL_NAME:
        llm = ChatOpenAI(model=MODEL_NAME, temperature = TEMPERATURE)
    else:
        try: # Reasoning Model에 한해 Path 분리
            llm = ChatOllama(model=MODEL_NAME, temperature = TEMPERATURE, reasoning = True)
            llm.invoke("hello!")
        except:
            llm = ChatOllama(model=MODEL_NAME, temperature = TEMPERATURE)

    print('## LLM 세팅 완료')
    Chroma().delete_collection() # 메모리 기존 데이터 삭제

    preprocessed_docs = load_docs_from_jsonl("eval.jsonl")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    print('## DB 구성 중...')

    chunks = text_splitter.split_documents(preprocessed_docs)
    print(f'## Chunks 생성 완료... 총 {len(chunks)} 개 청크')
    db = Chroma(embedding_function=emb_map[embedding_method],
                collection_metadata={'hnsw:space':'l2'},
                )

    for i in tqdm(range(0, len(chunks), 10)):
        db.add_documents(chunks[i:min(i+10, len(chunks))])

    print('## DB 구성 완료...')

    retriever = db.as_retriever(search_kwargs={'k':TOP_K})

    rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser())

    for query,reference in tqdm(zip(questions,ground_truths)):

        relevant_docs = [doc.page_content for doc in retriever.invoke(query)]
        response = rag_chain.invoke(query)
        dataset.append(
            {
                "user_input":query,
                "retrieved_contexts":relevant_docs,
                "response":response,
                "reference":reference
            }
        )

    print('## 답변 생성 완료...')
    evaluation_dataset = EvaluationDataset.from_list(dataset)

    result = evaluate(dataset=evaluation_dataset,
                    metrics=[BleuScore(), LLMContextRecall(), semantic_scorer, Faithfulness(), FactualCorrectness()],
                    llm=evaluator_llm,
                    embeddings = evaluator_embeddings
                    )

    result_str = str(result)
    result_dict = ast.literal_eval(result_str)
    return result_dict


실험 결과를 저장할 수 있습니다.

In [ ]:
import csv
import datetime
import os

result = RAG_pipeline()

timestamp = datetime.datetime.now().isoformat()
row = {
    "timestamp": timestamp,
    "model_name": MODEL_NAME,
    "embedding_method": embedding_method,
    "temperature": TEMPERATURE,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "top_k": TOP_K,
    **result
}

# 파일이 없으면 헤더 추가
write_header = not os.path.exists(csv_path)

with open(csv_path, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if write_header:
        writer.writeheader()
    writer.writerow(row)

print(f"[{timestamp}] {MODEL_NAME} {embedding_method}| temp={TEMPERATURE}, chunk={CHUNK_SIZE}/{CHUNK_OVERLAP}, top_k={TOP_K} | results={result}")
